In [1]:
import numpy as np
import pandas as pd

trans_df = pd.read_csv("datasets/LI-Small_Trans.csv")
trans_df.head(5)

,Timestamp,From Bank,Account,To Bank,Account.1,Amount Received,Receiving Currency,Amount Paid,Payment Currency,Payment Format,Is Laundering
0,2022/09/01 00:08,11,8000ECA90,11,8000ECA90,3195403.00,US Dollar,3195403.00,US Dollar,Reinvestment,0
1,2022/09/01 00:21,3402,80021DAD0,3402,80021DAD0,1858.96,US Dollar,1858.96,US Dollar,Reinvestment,0
2,2022/09/01 00:00,11,8000ECA90,1120,8006AA910,592571.00,US Dollar,592571.00,US Dollar,Cheque,0
3,2022/09/01 00:16,3814,8006AD080,3814,8006AD080,12.32,US Dollar,12.32,US Dollar,Reinvestment,0
4,2022/09/01 00:00,20,8006AD530,20,8006AD530,2941.56,US Dollar,2941.56,US Dollar,Reinvestment,0


In [2]:
# Analyze timestamps.
print(f"Timestamp range: [{trans_df["Timestamp"].min()},{trans_df["Timestamp"].max()}]")

Timestamp range: [2022/09/01 00:00,2022/09/17 15:28]


In [3]:
# Analyze transfers. Check for duplicate Account Numbers in different banks.
df_senders = trans_df[['From Bank', 'Account']].rename(columns={
    'From Bank': 'Bank', 
})
df_receivers = trans_df[['To Bank', 'Account.1']].rename(columns={
    'To Bank': 'Bank', 
    'Account.1': 'Account'
})
df_bank_accounts = pd.concat([df_senders, df_receivers],ignore_index=True)
df_bank_counts = df_bank_accounts.drop_duplicates().groupby('Account')['Bank'].count()
df_bank_counts[df_bank_counts > 1]

Account
817037B20    2
817038DF0    2
8177C8ED0    2
8177C94B0    2
Name: Bank, dtype: int64

In [4]:
#Filter non USD transactions.
trans_usd_df = trans_df[trans_df['Payment Currency'] == "US Dollar"]
print("SIZE:", trans_usd_df.shape[0])

SIZE: 2553887


In [5]:
# Analyze accounts.
accounts_df = pd.read_csv("datasets/LI-Small_accounts.csv")
print("SIZE:", accounts_df.shape[0])
accounts_df.head(5)

SIZE: 712688


,Bank Name,Bank ID,Account Number,Entity ID,Entity Name
0,China Bank #2820,314693,81B86A280,800D8CCF0,Corporation #41344
1,France Bank #4585,311253,8187FEA80,800B505E0,Corporation #54497
2,China Bank #2242,39996,803961E00,800D03F60,Partnership #36904
3,National Bank of Newport,331440,81B075800,801567C10,Corporation #16224
4,UK Bank #33,135417,80CF87C80,801085E00,Partnership #72930


In [6]:
trans_usd_sept_1st_df = trans_usd_df[(trans_usd_df["Timestamp"] >= '2022/09/01') & (trans_usd_df["Timestamp"] <= '2022/09/06')]
print("SIZE:", trans_usd_sept_1st_df.shape[0])

SIZE: 1393070


In [7]:
def filter_function(x):
    unique_account_size = x.groupby(["To Bank", "Account.1"]).size().size
    return  unique_account_size > 5 and unique_account_size < 10
ranged_trans_usd_sept_df = trans_usd_sept_1st_df.groupby(["From Bank", "Account"]).filter(filter_function)
print("SIZE:", ranged_trans_usd_sept_df.shape[0])

SIZE: 221273


In [8]:
#1. Cuenta de origen, cuenta de destino y monto para transacciones USD menores a 50.

low_profile_transactions = trans_usd_df[trans_usd_df["Amount Paid"] < 50].copy()
low_profile_transactions = low_profile_transactions[["From Bank", "Account", "To Bank", "Account.1", "Amount Paid"]]
low_profile_transactions

,From Bank,Account,To Bank,Account.1,Amount Paid
3,3814,8006AD080,3814,8006AD080,12.32
7,11,8000ECA90,11,8000ECA90,22.97
8,1120,8006AA910,243166,81470DCF0,43.53
9,1217,8006AD4E0,1217,8006AD4E0,5.04
10,224,8006AD580,23319,80567ED00,9.28
...,...,...,...,...,...
6923281,224,80011C480,224,80011C480,28.50
6923886,236488,814B886B1,236488,814B886B0,0.39
6924019,13474,803A93631,13474,803A93630,0.08
6924021,13474,803A93631,13474,803A93630,0.23


In [9]:
#2. Max amount by source bank, source Bank Id and Bank Name considering all the transactions.

max_amount_trans_usd_idx = trans_usd_df.groupby(["From Bank"])["Amount Paid"].idxmax()
max_amount_trans_usd = trans_usd_df.loc[max_amount_trans_usd_idx]
bank_name_by_bank_df = accounts_df[["Bank ID", "Bank Name"]].drop_duplicates(subset=["Bank ID"])
max_amount_bank = max_amount_trans_usd.merge(bank_name_by_bank_df, left_on="From Bank", right_on="Bank ID", how="left")
max_amount_bank["Bank Name"] = max_amount_bank.apply(lambda row: row["Bank Name"] if pd.notna(row["Bank Name"]) and str(row["Bank Name"]).strip() != "" else f"UNKNOWN_{int(row['From Bank'])}", axis=1)
max_amount_bank[["From Bank", "Account", "Bank Name", "Amount Paid"]]

,From Bank,Account,Bank Name,Amount Paid
0,0,801797710,Italy Bank #18,5.938537e+06
1,1,800908DF0,First Bank of Portland,5.280050e+08
2,2,8000859E0,India Bank #46,1.731930e+04
3,3,80570A290,Germany Bank #106,6.817745e+06
4,4,8062101E0,Japan Bank #40,1.749000e+01
...,...,...,...,...
15675,376905,81C1A5FA0,Bank of Tampa,7.699600e+02
15676,376932,81C1C9600,National Bank of New York,3.708250e+03
15677,376937,81C1CCBB0,National Bank of Portsmouth,2.322780e+03
15678,376938,81C1CCF10,Brownstone Savings Bank,2.280270e+03


In [10]:
#3. Cuenta de origen y monto de transacciones USD en [2022-09-06, 2022-09-15]
# con monto < 1% del promedio para el mismo Payment Format en [2022-09-01, 2022-09-05].

# Usamos límites por día inclusivos con ventanas [start, next_day) para no perder registros con hora.
trans_usd_base_period_df = trans_usd_df[(trans_usd_df["Timestamp"] >= "2022/09/01") & (trans_usd_df["Timestamp"] < "2022/09/06")]
trans_usd_eval_period_df = trans_usd_df[(trans_usd_df["Timestamp"] >= "2022/09/06") & (trans_usd_df["Timestamp"] < "2022/09/16")]

avg_amounts_per_type = trans_usd_base_period_df.groupby(["Payment Format"], as_index=False)["Amount Paid"].mean().rename(columns={"Amount Paid": "AVG"})

trans_usd_eval_with_avg_df = trans_usd_eval_period_df.merge(avg_amounts_per_type, on=["Payment Format"], how="inner")

lower_trans_usd_eval_with_avg_df = trans_usd_eval_with_avg_df[
    trans_usd_eval_with_avg_df["Amount Paid"] < trans_usd_eval_with_avg_df["AVG"] * 0.01
]

lower_trans_usd_eval_with_avg_df[["From Bank", "Account", "Amount Paid"]]

,From Bank,Account,Amount Paid
1,11495,8008AD900,10149.61
2,23,800077210,2680.03
3,23,800077210,10993.76
6,23,800077210,5845.00
7,23,800077210,8534.40
...,...,...,...
1160807,3389,807ECF700,13985.06
1160808,236488,814B886B1,0.39
1160809,13474,803A93631,0.08
1160810,13474,803A93631,0.23


In [11]:
#4. Cuentas que cumplen patrón scatter-gather con >= 5 cuentas intermedias
# distintas por par (A, B), para cuentas que enviaron en USD en [2022-09-01, 2022-09-05].

trans_usd_scatter_window_df = trans_usd_df[(trans_usd_df["Timestamp"] >= "2022/09/01") & (trans_usd_df["Timestamp"] < "2022/09/06")]

edges_df = trans_usd_scatter_window_df[["From Bank", "Account", "To Bank", "Account.1"]].drop_duplicates()

source_fanout_df = edges_df.groupby(["From Bank", "Account"]).size().reset_index(name="distinct_destinations")
candidate_sources_df = source_fanout_df[source_fanout_df["distinct_destinations"] >= 5][["From Bank", "Account"]]

first_hop_df = edges_df.merge(candidate_sources_df, on=["From Bank", "Account"], how="inner").rename(columns={
    "From Bank": "Source Bank",
    "Account": "Source Account",
    "To Bank": "Interm Bank",
    "Account.1": "Interm Account",
})

second_hop_df = edges_df.rename(columns={
    "From Bank": "Interm Bank",
    "Account": "Interm Account",
    "To Bank": "Final Bank",
    "Account.1": "Final Account",
})

paths_df = first_hop_df.merge(second_hop_df, on=["Interm Bank", "Interm Account"], how="inner")
paths_df = paths_df[(paths_df["Source Bank"] != paths_df["Final Bank"]) | (paths_df["Source Account"] != paths_df["Final Account"])]

paths_df = paths_df.copy()
paths_df["interm_key"] = paths_df["Interm Bank"].astype(str) + "_" + paths_df["Interm Account"]

interm_count_df = paths_df.groupby(
    ["Source Bank", "Source Account", "Final Bank", "Final Account"]
)["interm_key"].nunique().reset_index(name="n_intermediaries")

qualified_pairs_df = interm_count_df[interm_count_df["n_intermediaries"] >= 5][
    ["Source Bank", "Source Account", "Final Bank", "Final Account"]
]

scatter_gather_pairs_df = qualified_pairs_df.rename(columns={
    "Source Bank": "From Bank",
    "Source Account": "Account",
    "Final Bank": "To Bank",
    "Final Account": "Account.1",
}).reset_index(drop=True)
scatter_gather_pairs_df

,From Bank,Account,To Bank,Account.1
0,26,801A29190,20,800914620
1,1394,800701D70,11495,800A08E20


In [12]:
#5. Cantidad de transacciones del período [2022-09-01, 2022-09-05]
# con formato Wire o ACH cuyo monto convertido a USD sea menor a 1.
import json

# Only currencies present in cotizaciones.json; absent ones (Bitcoin) → NaN → discarded
CURRENCY_NAME_TO_CODE = {
    "Australian Dollar": "AUD",
    "Brazil Real":       "BRL",
    "Canadian Dollar":   "CAD",
    "Euro":              "EUR",
    "Mexican Peso":      "MXN",
    "Rupee":             "INR",
    "Ruble":             "RUB",
    "Saudi Riyal":       "SAR",
    "Shekel":            "ILS",
    "Swiss Franc":       "CHF",
    "UK Pound":          "GBP",
    "US Dollar":         None,   # base currency → rate = 1
    "Yen":               "JPY",
    "Yuan":              "CNY",
}

with open("datasets/cotizaciones.json") as f:
    cotizaciones_list = json.load(f)

# Rebuild { "YYYY-MM-DD": { "CODE": float } } from the flat list
raw_rates = {}
for entry in cotizaciones_list:
    raw_rates.setdefault(entry["date"], {})[entry["quote"]] = entry["rate"]

available_dates = sorted(raw_rates.keys())

def rates_for_date(date_str):
    """Most recent available rates on or before date_str (handles weekends)."""
    candidates = [d for d in available_dates if d <= date_str]
    return raw_rates[candidates[-1] if candidates else available_dates[0]]

# Build a tidy rates DataFrame: Date × Payment Currency → usd_rate
# JSON convention: 1 USD = X units of currency → 1 unit = 1/X USD
date_range = pd.date_range("2022-09-01", "2022-09-05", freq="D")
rates_rows = []
for dt in date_range:
    date_str = dt.strftime("%Y-%m-%d")
    day = rates_for_date(date_str)
    for name, code in CURRENCY_NAME_TO_CODE.items():
        usd_rate = 1.0 if code is None else 1.0 / day[code]
        rates_rows.append({"Date": date_str, "Payment Currency": name, "usd_rate": usd_rate})

rates_df = pd.DataFrame(rates_rows)

trans_sept_1st_df = trans_df[(trans_df["Timestamp"] >= "2022/09/01") & (trans_df["Timestamp"] < "2022/09/06")]
trans_sept_1st_wire_or_ach_df = trans_sept_1st_df[
    (trans_sept_1st_df["Payment Format"] == "Wire") | (trans_sept_1st_df["Payment Format"] == "ACH")
]
trans_sept_1st_wire_or_ach_converted_df = trans_sept_1st_wire_or_ach_df.copy()
trans_sept_1st_wire_or_ach_converted_df["Date"] = (
    pd.to_datetime(trans_sept_1st_wire_or_ach_converted_df["Timestamp"]).dt.strftime("%Y-%m-%d")
)
trans_sept_1st_wire_or_ach_converted_df = trans_sept_1st_wire_or_ach_converted_df.merge(
    rates_df, on=["Date", "Payment Currency"], how="left"
)
trans_sept_1st_wire_or_ach_converted_df["Amount"] = (
    trans_sept_1st_wire_or_ach_converted_df["Amount Paid"] * trans_sept_1st_wire_or_ach_converted_df["usd_rate"]
)
trans_sept_1st_wire_or_ach_lt_1_df = trans_sept_1st_wire_or_ach_converted_df[
    trans_sept_1st_wire_or_ach_converted_df["Amount"] < 1
]
print("SIZE:", trans_sept_1st_wire_or_ach_lt_1_df.shape[0])

SIZE: 9744
